# 26 Models: mixed models, GEE, GLS, curves, model averages, comparing models, choosing columns

The second half of the group "Models". This notebook holds only simplydata: data read with `sdata.table`, the words,
the numbers read back with simplydata's own words (`get_column`, `get_row`, `get_cell_at`, through `Result`), and the
comparison with `references/26_models_mixed_and_more.csv`; each comparison prints its "Source:" with a link.

References: first, numbers printed in the literature or the library's documentation, compared to half a unit of
their last digit: Bates, Maechler, Bolker & Walker (2015) for sleepstudy (the lme4 vignette) and the emmeans vignette
"basics" for the pigs and mtcars marginal means. Where nothing is printed, `scripts/references/26_models_mixed_and_more.py`
runs the help-page example of the library once (lme4, lmerTest, geepack, nlme, mgcv, splines, mice, SciPy bootstrap,
emmeans, anova, AICcmodavg, boot, glmnet) and saves its value with the link (relative 1e-10 for closed formulas, 1e-6
for iterative fits; same seed: 1e-12; multiple imputation against R mice: 4 Monte Carlo standard errors).

Section 12: the groups `show_rows_that_pull_model(grouped_by=)` gives back (from simplydata 3fd3b85 on; before, it only
printed them and this check waited for that word).

In [1]:
import simplydata as sdata

from reference_checks import Comparisons, Result, quiet

cmp = Comparisons("26_models_mixed_and_more")
DATA = "../data/"
FETCHED = "../data/fetched/"  # the datasets this repository does not host (written by scripts/get_data.py)

## 1. Mixed models on sleepstudy: own starting levels, own slopes, Satterthwaite

Bates et al. (2015) print `lmer(Reaction ~ Days + (Days | Subject))`: fixed effects 251.41 (6.82) and 10.47 (1.55).
`grouped_by="Subject"` alone is the random-intercept model (R lme4); `p_value_by="satterthwaite"` / `"few groups"`
against R lmerTest.

In [2]:
sleep = sdata.table(cmp.data_file("sleepstudy"))
sleep.explain("Reaction", using=["Days"], grouped_by="Subject", own_slope_for="Days")

Read "sleepstudy.csv" with "," between the columns (guessed). To be sure, give separator=",".


Explaining "Reaction" from "Days" (180 rows). Each effect below holds the other columns fixed:
  intercept: 251, the expected "Reaction" when every number column is 0.
  Days: each 1 more "Days" goes with 10.5 more "Reaction" (95 % CI 7.44 to 13.5); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
Method: linear mixed model with its own starting level and its own slope for "Days" for each "Subject", fitted by REML in R, as R lme4::lmer(Reaction ~ Days + (1 + Days | Subject)).
18 groups in "Subject". Variance of the starting levels between groups 612 (where the number columns are 0), within groups 655.
Variance of the own slopes between groups: "Days" 35.1. Correlation of starting level and slope: 0.0656.
P-values: Wald z-tests, with Wald 95 % intervals (standard errors as R vcov(), intervals as R confint(method = "Wald")); p_value_by="satterthwaite" gives Satterthwaite t-tests (R lmerTest) instead, which give slightly larger p-values when there a

,term,effect,95 % CI low,95 % CI high,standard error,statistic name,statistic,degrees of freedom,p-value
1,(intercept),251.405105,238.029141,264.781069,6.824597,z,36.838090,NaN,4.537104e-297
2,Days,10.467286,7.437594,13.496978,1.545790,z,6.771481,NaN,1.274703e-11


In [3]:
D = "sleepstudy (lme4)"
slope = Result(quiet(sleep.explain, "Reaction", using=["Days"], grouped_by="Subject", own_slope_for="Days"), key="term")
intercept = Result(quiet(sleep.explain, "Reaction", using=["Days"], grouped_by="Subject"), key="term")
satt = Result(quiet(sleep.explain, "Reaction", using=["Days"], grouped_by="Subject", p_value_by="satterthwaite"), key="term")
few = Result(quiet(sleep.explain, "Reaction", using=["Days"], grouped_by="Subject", p_value_by="few groups"), key="term")
for term in ("(intercept)", "Days"):
    cmp.check_all(D, 'explain (grouped_by=, own_slope_for="Days")',
                  {f"{term} estimate": slope[term, "effect"], f"{term} standard error": slope[term, "standard error"]})
    cmp.check_all(D, "explain (grouped_by=)", {f"{term} estimate": intercept[term, "effect"], f"{term} standard error": intercept[term, "standard error"]})
    for label, table in (('explain (grouped_by=, p_value_by="satterthwaite")', satt), ('explain (grouped_by=, p_value_by="few groups")', few)):
        cmp.check_all(D, label, {f"{term} degrees of freedom": table[term, "degrees of freedom"], f"{term} p-value": table[term, "p-value"]})

agrees  sleepstudy (lme4) | explain (grouped_by=, own_slope_for="Days") | (intercept) estimate: simplydata 251.405104848, Bates, Maechler, Bolker & Walker (2015), J. Stat. Softw. 67(1), section 3.6 and 5.2.4 251.41; difference 4.90e-03 (allowed 5e-03)
      Source: Bates, Maechler, Bolker & Walker (2015), J. Stat. Softw. 67(1), section 3.6 and 5.2.4 (printed) - https://doi.org/10.18637/jss.v067.i01 ; https://cran.r-project.org/web/packages/lme4/vignettes/lmer.pdf
agrees  sleepstudy (lme4) | explain (grouped_by=, own_slope_for="Days") | (intercept) standard error: simplydata 6.82459669495, Bates, Maechler, Bolker & Walker (2015), J. Stat. Softw. 67(1), section 3.6 and 5.2.4 6.82; difference 4.60e-03 (allowed 5e-03)
      Source: Bates, Maechler, Bolker & Walker (2015), J. Stat. Softw. 67(1), section 3.6 and 5.2.4 (printed) - https://doi.org/10.18637/jss.v067.i01 ; https://cran.r-project.org/web/packages/lme4/vignettes/lmer.pdf
agrees  sleepstudy (lme4) | explain (grouped_by=) | (interce

## 2. Around the mixed model: comparing models, the subjects' own levels, predictions

`compare_models(grouped_by=)` refits both models by maximum likelihood; Bates et al. (2015) print the ML fit of
`Reaction ~ Days + (1 | Subject)` (AIC 1802, BIC 1815, log-likelihood -897); the bigger model adds Days squared.
`find_group_levels_of(own_slope_for=)` against R `ranef(condVar = TRUE)` and `coef()`; `add_predicted_column(grouped_by=)`
against R `fitted()`.

In [4]:
cm = Result(quiet(sleep.compare_models, "Reaction", using=["Days"], against=["Days", "Days2"], grouped_by="Subject"))
small, big, test = cm.row(1), cm.row(2), cm.row(3)
cmp.check_all(D, "compare_models (grouped_by=)",
              {"AIC of Reaction ~ Days + (1 | Subject), ML": small["AIC"], "BIC of Reaction ~ Days + (1 | Subject), ML": small["BIC"],
               "log-likelihood of Reaction ~ Days + (1 | Subject), ML": small["log-likelihood"], "AIC bigger": big["AIC"],
               "BIC bigger": big["BIC"], "log-likelihood bigger": big["log-likelihood"], "chi-square": test["statistic"],
               "p-value": test["p-value"]})

levels = Result(quiet(sleep.find_group_levels_of, "Reaction", using=["Days"], grouped_by="Subject", own_slope_for="Days"))
subjects = levels["term"]
order = sorted(range(len(subjects)), key=lambda i: float(subjects[i]))  # subjects in R's order (by number)
for name, column in (("starting level", "starting level (distance from the average)"), ("slope", "slope for Days (distance from the average)"),
                     ("starting level conditional SD", "starting level conditional standard deviation"),
                     ("slope conditional SD", "slope for Days conditional standard deviation"),
                     ("own starting level", "own starting level"), ("own slope", "own slope for Days")):
    values = levels[column]
    cmp.check_vector(D, 'find_group_levels_of (own_slope_for="Days")', name, [values[i] for i in order])
predicted = sdata.table(cmp.data_file("sleepstudy"))
quiet(predicted.add_predicted_column, "Reaction", using=["Days"], grouped_by="Subject")
cmp.check_vector(D, "add_predicted_column (grouped_by=)", "predicted Reaction", Result(predicted)["predicted Reaction"])

agrees  sleepstudy (lme4) | compare_models (grouped_by=) | AIC of Reaction ~ Days + (1 | Subject), ML: simplydata 1802.07864301, Bates, Maechler, Bolker & Walker (2015), J. Stat. Softw. 67(1), section 3.6 and 5.2.4 (anova of fm3) 1802; difference 7.86e-02 (allowed 5e-01)
      Source: Bates, Maechler, Bolker & Walker (2015), J. Stat. Softw. 67(1), section 3.6 and 5.2.4 (anova of fm3) (printed) - https://doi.org/10.18637/jss.v067.i01 ; https://cran.r-project.org/web/packages/lme4/vignettes/lmer.pdf
agrees  sleepstudy (lme4) | compare_models (grouped_by=) | BIC of Reaction ~ Days + (1 | Subject), ML: simplydata 1814.85047041, Bates, Maechler, Bolker & Walker (2015), J. Stat. Softw. 67(1), section 3.6 and 5.2.4 (anova of fm3) 1815; difference 1.50e-01 (allowed 5e-01)
      Source: Bates, Maechler, Bolker & Walker (2015), J. Stat. Softw. 67(1), section 3.6 and 5.2.4 (anova of fm3) (printed) - https://doi.org/10.18637/jss.v067.i01 ; https://cran.r-project.org/web/packages/lme4/vignettes/lme

agrees  sleepstudy (lme4) | find_group_levels_of (own_slope_for="Days") | starting level [1]: simplydata 2.2585509499, R lme4 ranef(condVar = TRUE) and coef (lme4 2.1.0) 2.2585509499; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lme4 ranef(condVar = TRUE) and coef (lme4 2.1.0) (run reference) - https://search.r-project.org/CRAN/refmans/lme4/html/ranef.html
agrees  sleepstudy (lme4) | find_group_levels_of (own_slope_for="Days") | starting level [2]: simplydata -40.3987380788, R lme4 ranef(condVar = TRUE) and coef (lme4 2.1.0) -40.3987380788; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lme4 ranef(condVar = TRUE) and coef (lme4 2.1.0) (run reference) - https://search.r-project.org/CRAN/refmans/lme4/html/ranef.html
agrees  sleepstudy (lme4) | find_group_levels_of (own_slope_for="Days") | starting level [3]: simplydata -38.9604089576, R lme4 ranef(condVar = TRUE) and coef (lme4 2.1.0) -38.9604089576; relative difference 0.00e+00 (allowed 1e-06)
      Source:

## 3. A yes/no mixed model: cbpp

The help page of `lme4::glmer`: `glmer(cbind(incidence, size - incidence) ~ period + (1 | herd), family = binomial)`.

In [5]:
cbpp = sdata.table(cmp.data_file("cbpp"))
quiet(cbpp.turn_into_text, "period")
gm = Result(quiet(cbpp.explain, "incidence", using=["period"], outcome="yes or no", out_of="size", grouped_by="herd"), key="term")
for term in ["(intercept)", "period (2)", "period (3)", "period (4)"]:
    for column in ("log odds ratio", "standard error", "p-value"):
        cmp.check("cbpp (lme4)", 'explain (outcome="yes or no", out_of=, grouped_by=)', f"{term} {column}", gm[term, column])

Read "cbpp.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  cbpp (lme4) | explain (outcome="yes or no", out_of=, grouped_by=) | (intercept) log odds ratio: simplydata -1.39853204355, R lme4 glmer (help page example) (lme4 2.1.0) -1.39853204355; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lme4 glmer (help page example) (lme4 2.1.0) (run reference) - https://search.r-project.org/CRAN/refmans/lme4/html/glmer.html
agrees  cbpp (lme4) | explain (outcome="yes or no", out_of=, grouped_by=) | (intercept) standard error: simplydata 0.232473324921, R lme4 glmer (help page example) (lme4 2.1.0) 0.232473324921; relative difference 1.19e-16 (allowed 1e-06)
      Source: R lme4 glmer (help page example) (lme4 2.1.0) (run reference) - https://search.r-project.org/CRAN/refmans/lme4/html/glmer.html
agrees  cbpp (lme4) | explain (outcome="yes or no", out_of=, grouped_by=) | (intercept) p-value: simplydata 1.78910090372e-09, R lme4 glmer (help page example) (lme4 2.1.0) 1.78910090372e-09; relative difference 2.20e-15 (allowed 1e-06)
     

## 4. Effects for the whole population (GEE): pig weights (dietox)

`Weight ~ Time + Cu` with the three working correlations, against R `geepack::geeglm` (whose help page uses dietox).

In [6]:
diet = sdata.table(cmp.data_file("dietox"))
for within in ("alike", "in time order", "independent"):
    ge = Result(quiet(diet.explain, "Weight", using=["Time", "Cu"], grouped_by="Pig", effects="for the whole population",
                      within_groups=within), key="term")
    for term in ["(intercept)", "Time", "Cu (Cu035)", "Cu (Cu175)"]:
        cmp.check_all("dietox (geepack)", f'explain (effects="for the whole population", within_groups="{within}")',
                      {f"{term} estimate": ge[term, "effect"], f"{term} standard error": ge[term, "standard error"]})

Read "dietox.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  dietox (geepack) | explain (effects="for the whole population", within_groups="alike") | (intercept) estimate: simplydata 15.422374318, R geepack geeglm (geepack 1.3.13) 15.422374318; relative difference 1.96e-15 (allowed 1e-06)
      Source: R geepack geeglm (geepack 1.3.13) (run reference) - https://search.r-project.org/CRAN/refmans/geepack/html/geeglm.html
agrees  dietox (geepack) | explain (effects="for the whole population", within_groups="alike") | (intercept) standard error: simplydata 1.0250354471, R geepack geeglm (geepack 1.3.13) 1.0250354471; relative difference 1.08e-14 (allowed 1e-06)
      Source: R geepack geeglm (geepack 1.3.13) (run reference) - https://search.r-project.org/CRAN/refmans/geepack/html/geeglm.html
agrees  dietox (geepack) | explain (effects="for the whole population", within_groups="alike") | Time estimate: simplydata 6.94252038338, R geepack geeglm (geepack 

## 5. Generalised least squares: mares' follicles (Ovary)

The model of the `nlme::gls` help page, with AR-1 over the rows in table order and a separate spread for each mare.

In [7]:
ovary = sdata.table(cmp.data_file("Ovary"))
for option, value in (("neighbours_alike_in_order_of", "order"), ("spread_changes_with", "Mare")):
    gl = Result(quiet(ovary.explain, "follicles", using=["s", "c"], **{option: value}), key="term")
    ours = {f"{t} {q}": gl[t, c] for t in ["(intercept)", "s", "c"] for q, c in (("estimate", "effect"), ("standard error", "standard error"),
                                                                                   ("p-value", "p-value"))}
    if option == "neighbours_alike_in_order_of":
        ours["correlation of neighbours"] = gl['correlation of neighbours in "order" order', "effect"]
    cmp.check_all("Ovary (nlme)", f'explain ({option}="{value}")', ours)

Read "Ovary.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  Ovary (nlme) | explain (neighbours_alike_in_order_of="order") | (intercept) estimate: simplydata 12.2383282306, R nlme gls (nlme 3.1.172) 12.2383282306; relative difference 0.00e+00 (allowed 1e-06)
      Source: R nlme gls (nlme 3.1.172) (run reference) - https://search.r-project.org/CRAN/refmans/nlme/html/gls.html
agrees  Ovary (nlme) | explain (neighbours_alike_in_order_of="order") | (intercept) standard error: simplydata 0.68721921336, R nlme gls (nlme 3.1.172) 0.68721921336; relative difference 0.00e+00 (allowed 1e-06)
      Source: R nlme gls (nlme 3.1.172) (run reference) - https://search.r-project.org/CRAN/refmans/nlme/html/gls.html
agrees  Ovary (nlme) | explain (neighbours_alike_in_order_of="order") | (intercept) p-value: simplydata 3.90829629066e-49, R nlme gls (nlme 3.1.172) 3.90829629066e-49; relative difference 0.00e+00 (allowed 1e-06)
      Source: R nlme gls (nlme 3.1.172) (r

agrees  Ovary (nlme) | explain (spread_changes_with="Mare") | (intercept) estimate: simplydata 12.32047764, R nlme gls (nlme 3.1.172) 12.32047764; relative difference 0.00e+00 (allowed 1e-06)
      Source: R nlme gls (nlme 3.1.172) (run reference) - https://search.r-project.org/CRAN/refmans/nlme/html/gls.html
agrees  Ovary (nlme) | explain (spread_changes_with="Mare") | (intercept) standard error: simplydata 0.227994069364, R nlme gls (nlme 3.1.172) 0.227994069364; relative difference 1.22e-16 (allowed 1e-06)
      Source: R nlme gls (nlme 3.1.172) (run reference) - https://search.r-project.org/CRAN/refmans/nlme/html/gls.html
agrees  Ovary (nlme) | explain (spread_changes_with="Mare") | (intercept) p-value: simplydata 3.03620535298e-158, R nlme gls (nlme 3.1.172) 3.03620535298e-158; relative difference 0.00e+00 (allowed 1e-06)
      Source: R nlme gls (nlme 3.1.172) (run reference) - https://search.r-project.org/CRAN/refmans/nlme/html/gls.html
agrees  Ovary (nlme) | explain (spread_cha

## 6. Curves inside a model: a GAM (mcycle) and a natural spline (cars)

In [8]:
gam = Result(quiet(sdata.table(cmp.data_file("mcycle")).explain, "accel", using=["times"], curve_for="times"), key="term")
cmp.check_all("mcycle (MASS)", 'explain (curve_for="times")',
              {"intercept": gam["(intercept)", "effect"], "intercept standard error": gam["(intercept)", "standard error"],
               "effective degrees of freedom of s(times)": gam["s(times)", "effective degrees of freedom"], "F of s(times)": gam["s(times)", "statistic"]})
ns = Result(quiet(sdata.table(cmp.data_file("cars")).explain, "dist", using=["speed"], curve_for="speed", with_knots=3), key="term")
cmp.check_all("cars", 'explain (curve_for="speed", with_knots=3)',
              {"F of the whole curve": ns["speed (whole curve)", "statistic"], "p-value of the whole curve": ns["speed (whole curve)", "p-value"]})

Read "mcycle.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  mcycle (MASS) | explain (curve_for="times") | intercept: simplydata -25.5458646617, R mgcv gam (REML) (mgcv 1.9.5) -25.5458646617; relative difference 0.00e+00 (allowed 1e-06)
      Source: R mgcv gam (REML) (mgcv 1.9.5) (run reference) - https://search.r-project.org/CRAN/refmans/mgcv/html/gam.html
agrees  mcycle (MASS) | explain (curve_for="times") | intercept standard error: simplydata 1.9511957193, R mgcv gam (REML) (mgcv 1.9.5) 1.9511957193; relative difference 0.00e+00 (allowed 1e-06)
      Source: R mgcv gam (REML) (mgcv 1.9.5) (run reference) - https://search.r-project.org/CRAN/refmans/mgcv/html/gam.html
agrees  mcycle (MASS) | explain (curve_for="times") | effective degrees of freedom of s(times): simplydata 8.62469099939, R mgcv gam (REML) (mgcv 1.9.5) 8.62469099939; relative difference 0.00e+00 (allowed 1e-06)
      Source: R mgcv gam (REML) (mgcv 1.9.5) (run reference) - https://search.r-project.org/CRAN/refmans/mgcv/html/gam.html
agrees  mcycle (MASS) | explain (cur

## 7. Multiple imputation: nhanes (mice)

simplydata fills the empty cells 20 times (statsmodels MICEData); R mice draws differently, so the pooled estimates
are compared with R `mice(m = 400)` + `pool()` within 4 Monte Carlo standard errors; a rerun with the same
`with_seed=` must agree exactly.

In [9]:
nhanes = sdata.table(cmp.data_file("nhanes"))
mi = Result(quiet(nhanes.explain, "chl", using=["age", "bmi"], empty_cells="multiple imputation"), key="term")
again = Result(quiet(nhanes.explain, "chl", using=["age", "bmi"], empty_cells="multiple imputation", with_seed=1), key="term")
for term in ["(intercept)", "age", "bmi"]:
    cmp.check("nhanes (mice)", 'explain (empty_cells="multiple imputation", with_seed=)', f"{term} pooled estimate", mi[term, "effect"])
    cmp.check_equal("nhanes (mice)", 'explain (empty_cells="multiple imputation", with_seed=)', f"{term} estimate, rerun with with_seed=1",
                    mi[term, "effect"], again[term, "effect"], "simplydata, same seed")

Read "nhanes.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  nhanes (mice) | explain (empty_cells="multiple imputation", with_seed=) | (intercept) pooled estimate: simplydata -25.1611334113, R mice (m = 400) + pool, within 4 MCSE (mice 3.19.0) -14.1493854859; difference 1.10e+01 (allowed 4e+01)
      Source: R mice (m = 400) + pool, within 4 MCSE (mice 3.19.0) (run reference) - https://search.r-project.org/CRAN/refmans/mice/html/mice.html
agrees  nhanes (mice) | explain (empty_cells="multiple imputation", with_seed=) | (intercept) estimate, rerun with with_seed=1: simplydata -25.1611334113, simplydata, same seed -25.1611334113; relative difference 0.00e+00 (allowed 1e-12)
      Source: simplydata, same seed
agrees  nhanes (mice) | explain (empty_cells="multiple imputation", with_seed=) | age pooled estimate: simplydata 35.1812101885, R mice (m = 400) + pool, within 4 MCSE (mice 3.19.0) 32.5900276947; difference 2.59e+00 (allowed 6e+00)
      Source: R mice (m = 400) + pool, within 4 MCSE (mice 3.19.0) (run reference) - https://search.r-p

## 8. Bootstrap intervals with a seed: stack loss

`interval_by="bootstrap", resamples=2000, with_seed=7`: BCa intervals against SciPy `stats.bootstrap` with the same seed.

In [10]:
stackloss = sdata.table(DATA + "stackloss.csv")
bt = Result(quiet(stackloss.explain, "stack_loss", using=["air_flow"], interval_by="bootstrap", resamples=2000, with_seed=7), key="term")
for term in ["(intercept)", "air_flow"]:
    cmp.check_all("stackloss", 'explain (interval_by="bootstrap", resamples=, with_seed=)',
                  {f"{term} 95 % low": bt[term, "95 % CI low"], f"{term} 95 % high": bt[term, "95 % CI high"]})

Read "stackloss.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  stackloss | explain (interval_by="bootstrap", resamples=, with_seed=) | (intercept) 95 % low: simplydata -52.7232696877, SciPy stats.bootstrap (BCa, same seed) (scipy 1.18.1) -52.7232696877; relative difference 1.08e-15 (allowed 1e-12)
      Source: SciPy stats.bootstrap (BCa, same seed) (scipy 1.18.1) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  stackloss | explain (interval_by="bootstrap", resamples=, with_seed=) | (intercept) 95 % high: simplydata -23.6441154507, SciPy stats.bootstrap (BCa, same seed) (scipy 1.18.1) -23.6441154507; relative difference 7.06e-15 (allowed 1e-12)
      Source: SciPy stats.bootstrap (BCa, same seed) (scipy 1.18.1) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  stackloss | explain (interval_by="bootstrap", resamples=, with_seed=) | air_flow 95 % low: simplydata 0.611520732563, SciPy stats.bootstrap (BCa, same seed) 

## 9. Model averages: `find_model_averages_of`

Printed in the emmeans vignette "basics": pigs `lm(inverse(conc) ~ source + factor(percent))` by source and by percent,
and mtcars `lm(mpg ~ factor(cyl) + disp + dispsq)`. Run once: the help-page example on warpbreaks with `pairs()`,
the `comparing=` choices, `within_each_group_in=`, `over_range_of=`, and other outcomes.

In [11]:
pigs = sdata.table(cmp.data_file("pigs"))
by_source = Result(quiet(pigs.find_model_averages_of, "inverse_conc", for_each_group_in="source", using=["source", "percent_group"]), key="term")
for g in ("fish", "soy", "skim"):
    cmp.check_all("pigs (emmeans)", "find_model_averages_of", {f"{g} model average": by_source[g, "model average"],
                                                               f"{g} standard error": by_source[g, "standard error"],
                                                               f"{g} degrees of freedom": by_source[g, "degrees of freedom"]})
by_percent = Result(quiet(pigs.find_model_averages_of, "inverse_conc", for_each_group_in="percent_group", using=["source", "percent_group"]),
                    key="term")
for g in ("09", "12", "15", "18"):
    cmp.check_all("pigs (emmeans)", "find_model_averages_of", {f"percent {g} model average": by_percent[f"p{g}", "model average"],
                                                               f"percent {g} standard error": by_percent[f"p{g}", "standard error"]})
cars_by_cyl = Result(quiet(sdata.table(cmp.data_file("mtcars")).find_model_averages_of, "mpg", for_each_group_in="cylinders",
                           using=["cylinders", "disp", "dispsq"]), key="term")
for g in ("4", "6", "8"):
    cmp.check_all("mtcars", "find_model_averages_of", {f"{g} cylinders model average": cars_by_cyl[f"c{g}", "model average"],
                                                       f"{g} cylinders standard error": cars_by_cyl[f"c{g}", "standard error"]})

Read "pigs.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  pigs (emmeans) | find_model_averages_of | fish model average: simplydata 0.033688989962, emmeans vignette 'basics' (Lenth), printed output 0.0337; difference 1.10e-05 (allowed 5e-05)
      Source: emmeans vignette 'basics' (Lenth), printed output (printed) - https://cran.r-project.org/web/packages/emmeans/vignettes/basics.html
agrees  pigs (emmeans) | find_model_averages_of | fish standard error: simplydata 0.000926036086537, emmeans vignette 'basics' (Lenth), printed output 0.000926; difference 3.61e-08 (allowed 5e-07)
      Source: emmeans vignette 'basics' (Lenth), printed output (printed) - https://cran.r-project.org/web/packages/emmeans/vignettes/basics.html
agrees  pigs (emmeans) | find_model_averages_of | fish degrees of freedom: simplydata 23, emmeans vignette 'basics' (Lenth), printed output 23; difference 0.00e+00 (allowed 5e-01)
      Source: emmeans vignette 'basics' (Lenth), pri

In [12]:
warp = sdata.table(DATA + "warpbreaks.csv")
ew = Result(quiet(warp.find_model_averages_of, "breaks", for_each_group_in="tension", using=["wool * tension"]), key="term")
ours = {f"{t} model average": ew[t, "model average"] for t in "LMH"}
ours.update({f"{t} standard error": ew[t, "standard error"] for t in "LMH"})
for pair in ("H - L", "H - M", "L - M"):
    ours.update({f"{pair} difference": ew[pair, "difference"], f"{pair} p-value (Tukey)": ew[pair, "p-value (Tukey)"]})
cmp.check_all("warpbreaks", "find_model_averages_of (wool * tension)", ours)

for comparing in ("each with the first", "each with the next", "each with the average", "trend", "each with the average of the ones before"):
    t = Result(quiet(warp.find_model_averages_of, "breaks", for_each_group_in="tension", using=["wool", "tension"], comparing=comparing,
                     in_order=["L", "M", "H"]))
    estimate = "difference" if "difference" in t.columns else "contrast"
    pcol = [c for c in t.columns if c.startswith("p-value")][0]
    contrast = 0
    for i in range(1, len(t) + 1):
        row = t.row(i)
        if row["model average"] is not None and row["model average"] == row["model average"]:  # a group's own average
            continue
        contrast += 1
        ours = {f"contrast {contrast} estimate": row[estimate], f"contrast {contrast} standard error": row["standard error"]}
        if comparing in ("trend", "each with the average of the ones before"):
            ours[f"contrast {contrast} p-value"] = row[pcol]
        cmp.check_all("warpbreaks", f'find_model_averages_of (comparing="{comparing}")', ours)

wi = Result(quiet(warp.find_model_averages_of, "breaks", for_each_group_in="tension", using=["wool * tension"], within_each_group_in="wool"),
            key="term")
for w in "AB":
    for t in "LMH":
        cmp.check_all("warpbreaks", 'find_model_averages_of (within_each_group_in="wool")',
                      {f"{t} (within {w}) model average": wi[f"{t} (within {w})", "model average"],
                       f"{t} (within {w}) standard error": wi[f"{t} (within {w})", "standard error"]})
over = Result(quiet(sdata.table(cmp.data_file("mtcars")).find_model_averages_of, "mpg", for_each_group_in="cylinders", using=["cylinders", "wt"],
                    over_range_of="wt"))
cmp.check_vector("mtcars", 'find_model_averages_of (over_range_of="wt")', "model average", over["model average"])
cmp.check_vector("mtcars", 'find_model_averages_of (over_range_of="wt")', "standard error", over["standard error"])

Read "warpbreaks.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  warpbreaks | find_model_averages_of (wool * tension) | L model average: simplydata 36.3888888889, R emmeans + pairs (help page example) (emmeans 2.0.5) 36.3888888889; relative difference 0.00e+00 (allowed 1e-10)
      Source: R emmeans + pairs (help page example) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  warpbreaks | find_model_averages_of (wool * tension) | M model average: simplydata 26.3888888889, R emmeans + pairs (help page example) (emmeans 2.0.5) 26.3888888889; relative difference 5.39e-16 (allowed 1e-10)
      Source: R emmeans + pairs (help page example) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  warpbreaks | find_model_averages_of (wool * tension) | H model average: simplydata 21.6666666667, R emmeans + pairs (help page example) (emmeans 2.0.5) 21.6666666667; relative difference 4.92e-16 (allowed 1e-10)
      Source: R emmeans + pairs 

agrees  warpbreaks | find_model_averages_of (comparing="each with the average of the ones before") | contrast 1 estimate: simplydata -10, R emmeans contrast (emmeans 2.0.5) -10; relative difference 0.00e+00 (allowed 1e-10)
      Source: R emmeans contrast (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/contrast.html
agrees  warpbreaks | find_model_averages_of (comparing="each with the average of the ones before") | contrast 1 standard error: simplydata 3.87237764713, R emmeans contrast (emmeans 2.0.5) 3.87237764713; relative difference 1.15e-16 (allowed 1e-10)
      Source: R emmeans contrast (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/contrast.html
agrees  warpbreaks | find_model_averages_of (comparing="each with the average of the ones before") | contrast 1 p-value: simplydata 0.0127868279209, R emmeans contrast (emmeans 2.0.5) 0.0127868279209; relative difference 7.73e-15 (allowed 1e-10)
      Sour

agrees  warpbreaks | find_model_averages_of (within_each_group_in="wool") | L (within A) model average: simplydata 44.5555555556, R emmeans (~ tension | wool) (emmeans 2.0.5) 44.5555555556; relative difference 1.59e-16 (allowed 1e-10)
      Source: R emmeans (~ tension | wool) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  warpbreaks | find_model_averages_of (within_each_group_in="wool") | L (within A) standard error: simplydata 3.64676134574, R emmeans (~ tension | wool) (emmeans 2.0.5) 3.64676134574; relative difference 1.22e-16 (allowed 1e-10)
      Source: R emmeans (~ tension | wool) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  warpbreaks | find_model_averages_of (within_each_group_in="wool") | M (within A) model average: simplydata 24, R emmeans (~ tension | wool) (emmeans 2.0.5) 24; relative difference 2.96e-16 (allowed 1e-10)
      Source: R emmeans (~

Other outcomes, against R emmeans on the model fitted directly (response scale).

In [13]:
CASES = [("birthwt", sdata.table(cmp.data_file("birthwt")), "low", "race_group", ["race_group", "lwt"], {"outcome": "yes or no"}, ["black", "other", "white"]),
         ("NMES 1988", sdata.table(FETCHED + "nmes1988.csv", separator=","), "visits", "health", ["health", "chronic"], {"outcome": "counts"}, ["average", "excellent", "poor"]),
         ("NMES 1988", sdata.table(FETCHED + "nmes1988.csv", separator=","), "visits", "health", ["health", "chronic"], {"outcome": "counts (overdispersed)"},
          ["average", "excellent", "poor"]),
         ("NMES 1988", sdata.table(FETCHED + "nmes1988.csv", separator=","), "visits", "health", ["health", "chronic"], {"outcome": "counts that vary a lot"},
          ["average", "excellent", "poor"]),
         ("GasolineYield (betareg)", sdata.table(cmp.data_file("GasolineYield")), "yield", "batch", ["batch", "temp"], {"outcome": "shares"},
          [f"b{k:02d}" for k in range(1, 11)]),
         ("cbpp (lme4), one row per animal", sdata.table(cmp.data_file("cbpp_one_row_per_animal")), "sick", "period", ["period"],
          {"outcome": "yes or no", "grouped_by": "herd"}, ["1", "2", "3", "4"])]
for dataset, data, column, groups, using, options, levels in CASES:
    if groups == "period":
        quiet(data.turn_into_text, "period")
        quiet(data.turn_into_text, "herd")
    t = Result(quiet(data.find_model_averages_of, column, for_each_group_in=groups, using=using, **options), key="term")
    label = "find_model_averages_of (" + ", ".join(f'{k}="{v}"' for k, v in options.items()) + ")"
    for g in levels:
        cmp.check_all(dataset, label, {f"{g} model average": t[g, "model average"], f"{g} standard error": t[g, "standard error"]})

housing = sdata.table(cmp.data_file("housing_one_row_per_tenant"))
for outcome in ("ordered groups", "groups"):
    t = quiet(housing.find_model_averages_of, "Sat", for_each_group_in="Infl", using=["Infl", "Type", "Cont"], outcome=outcome,
              in_order=["Low", "Medium", "High"] if outcome == "ordered groups" else None)
    for infl in ("High", "Low", "Medium"):
        for sat in ("Low", "Medium", "High"):
            hit = Result(t, where={"term": infl, "Sat group": sat})
            cmp.check_all("housing (MASS)", f'find_model_averages_of (outcome="{outcome}")',
                          {f"chance of {sat} for Infl {infl}": hit.cell(1, "model chance"),
                           f"standard error of the chance of {sat} for Infl {infl}": hit.cell(1, "standard error")})

Read "birthwt.csv" with "," between the columns (guessed). To be sure, give separator=",".
Read "GasolineYield.csv" with "," between the columns (guessed). To be sure, give separator=",".
Read "cbpp_one_row_per_animal.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  birthwt | find_model_averages_of (outcome="yes or no") | black model average: simplydata 0.477673717487, R emmeans (type = response) (emmeans 2.0.5) 0.477673717487; relative difference 1.16e-16 (allowed 1e-06)
      Source: R emmeans (type = response) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  birthwt | find_model_averages_of (outcome="yes or no") | black standard error: simplydata 0.104330408926, R emmeans (type = response) (emmeans 2.0.5) 0.104330408926; relative difference 2.66e-16 (allowed 1e-06)
      Source: R emmeans (type = response) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  birthwt | find_model_averages_of (outcome="yes or no") | other model average: simplydata 0.334071451099, R emmeans (type = response) (emmeans 2.0.5) 0.334071451099; relative difference 0.00e+00 (allowed 1e-06)
      Source: R emmeans (type = response) (emmeans 2.0.5)

agrees  NMES 1988 | find_model_averages_of (outcome="counts that vary a lot") | average model average: simplydata 5.43763761054, R emmeans (type = response) (emmeans 2.0.5) 5.43763761054; relative difference 0.00e+00 (allowed 1e-06)
      Source: R emmeans (type = response) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  NMES 1988 | find_model_averages_of (outcome="counts that vary a lot") | average standard error: simplydata 0.0955026342849, R emmeans (type = response) (emmeans 2.0.5) 0.0955026342849; relative difference 7.27e-16 (allowed 1e-06)
      Source: R emmeans (type = response) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  NMES 1988 | find_model_averages_of (outcome="counts that vary a lot") | excellent model average: simplydata 3.86493987589, R emmeans (type = response) (emmeans 2.0.5) 3.86493987589; relative difference 2.30e-16 (allowed 1e-06)
      

agrees  GasolineYield (betareg) | find_model_averages_of (outcome="shares") | b01 model average: simplydata 0.312209080865, R emmeans (type = response) (emmeans 2.0.5) 0.312209080865; relative difference 1.78e-16 (allowed 1e-06)
      Source: R emmeans (type = response) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  GasolineYield (betareg) | find_model_averages_of (outcome="shares") | b01 standard error: simplydata 0.0120084856063, R emmeans (type = response) (emmeans 2.0.5) 0.0120084856063; relative difference 2.23e-13 (allowed 1e-06)
      Source: R emmeans (type = response) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  GasolineYield (betareg) | find_model_averages_of (outcome="shares") | b02 model average: simplydata 0.232375914427, R emmeans (type = response) (emmeans 2.0.5) 0.232375914427; relative difference 1.19e-15 (allowed 1e-06)
      Source: R emmean

explained difference – machine (B25)  cbpp (lme4), one row per animal | find_model_averages_of (outcome="yes or no", grouped_by="herd") | 1 model average: simplydata 0.198050871617, R emmeans (type = response) (emmeans 2.0.5) 0.198049157244; relative difference 8.66e-06 (allowed 1e-06)
      why: Two glmer fits of the same model agree to about 1e-5 - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#b25-two-glmer-fits-of-the-same-model-agree-to-about-1e-5
      Source: R emmeans (type = response) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
explained difference – machine (B25)  cbpp (lme4), one row per animal | find_model_averages_of (outcome="yes or no", grouped_by="herd") | 1 standard error: simplydata 0.0369226631586, R emmeans (type = response) (emmeans 2.0.5) 0.0369227565089; relative difference 2.53e-06 (allowed 1e-06)
      why: Two glmer fits of the same model agree to about 1e-5 - https://github.com

agrees  housing (MASS) | find_model_averages_of (outcome="groups") | chance of Low for Infl High: simplydata 0.205185965989, R emmeans (mode = prob) (emmeans 2.0.5) 0.205186110113; relative difference 7.02e-07 (allowed 1e-06)
      Source: R emmeans (mode = prob) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  housing (MASS) | find_model_averages_of (outcome="groups") | standard error of the chance of Low for Infl High: simplydata 0.0206535215386, R emmeans (mode = prob) (emmeans 2.0.5) 0.0206535271922; relative difference 2.74e-07 (allowed 1e-06)
      Source: R emmeans (mode = prob) (emmeans 2.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/emmeans/html/emmeans.html
agrees  housing (MASS) | find_model_averages_of (outcome="groups") | chance of Medium for Infl High: simplydata 0.230029002256, R emmeans (mode = prob) (emmeans 2.0.5) 0.230028930826; relative difference 3.11e-07 (allowed 1e-06)
      Source: R

## 10. Comparing models

Nested models (F test for numbers and "counts (overdispersed)", likelihood-ratio test for the others), AICc with Akaike
weights (R `AICcmodavg::aictab`), and the 10-fold cross-validation error after `set.seed(with_seed)` (R `boot::cv.glm`,
whose help page uses the mammals data).

In [14]:
cmp_numbers = Result(quiet(stackloss.compare_models, "stack_loss", using=["air_flow"], against=["air_flow", "water_temp", "acid_conc"]))
cmp.check_all("stackloss", 'compare_models (outcome="numbers")', {"AIC smaller": cmp_numbers.cell(1, "AIC"), "AIC bigger": cmp_numbers.cell(2, "AIC"),
                                                                  "statistic": cmp_numbers.cell(3, "statistic"), "p-value": cmp_numbers.cell(3, "p-value")})
births = sdata.table(cmp.data_file("birthwt"))
NESTED = [("NMES 1988", sdata.table(FETCHED + "nmes1988.csv", separator=","), "visits", ["hospital"], ["hospital", "chronic"], "counts"),
          ("NMES 1988", sdata.table(FETCHED + "nmes1988.csv", separator=","), "visits", ["hospital"], ["hospital", "chronic"], "counts (overdispersed)"),
          ("NMES 1988", sdata.table(FETCHED + "nmes1988.csv", separator=","), "visits", ["hospital"], ["hospital", "chronic"], "counts that vary a lot"),
          ("birthwt", births, "low", ["lwt"], ["lwt", "smoke"], "yes or no"),
          ("birthwt weight", births, "bwt_kg", ["lwt"], ["lwt", "smoke"], "positive numbers"),
          ("bioChemists (pscl)", sdata.table(cmp.data_file("bioChemists")), "art", ["ment"], ["ment", "kid5"], "counts with many zeros")]
for dataset, data, column, using, against, outcome in NESTED:
    c = Result(quiet(data.compare_models, column, using=using, against=against, outcome=outcome))
    cmp.check_all(dataset, f'compare_models (outcome="{outcome}")', {"statistic": c.cell(3, "statistic"), "p-value": c.cell(3, "p-value")})

mammals = sdata.table(cmp.data_file("mammals"))
ac = Result(quiet(mammals.compare_models, "log_brain", using=[["log_body"], ["log_body", "log_body2"]]), key="term")
for name in ac.index:
    cmp.check_all("mammals (MASS)", "compare_models (using= a list of models: AICc)",
                  {f"{name}: {c}": ac[name, c] for c in ("AIC", "AICc", "difference in AICc", "Akaike weight", "log-likelihood")})
for seed in (1, 7):
    cv = Result(quiet(mammals.compare_models, "log_brain", using=["log_body"], against=["log_body", "log_body2"], check_on_left_out_rows=True,
                      with_seed=seed), key="term")
    for name in cv.index:
        cmp.check_all("mammals (MASS)", f"compare_models (check_on_left_out_rows=True, with_seed={seed})",
                      {f"{name}: error on left-out rows": cv[name, "mean squared error on left-out rows"],
                       f"{name}: error on left-out rows (bias-corrected)": cv[name, "mean squared error on left-out rows (bias-corrected)"]})

agrees  stackloss | compare_models (outcome="numbers") | AIC smaller: simplydata 122.7371023, R anova(m0, m1) and AIC (stats 4.6.2) 122.7371023; relative difference 0.00e+00 (allowed 1e-10)
      Source: R anova(m0, m1) and AIC (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/anova.lm.html
agrees  stackloss | compare_models (outcome="numbers") | AIC bigger: simplydata 114.575591005, R anova(m0, m1) and AIC (stats 4.6.2) 114.575591005; relative difference 0.00e+00 (allowed 1e-10)
      Source: R anova(m0, m1) and AIC (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/anova.lm.html
agrees  stackloss | compare_models (outcome="numbers") | statistic: simplydata 6.66796668331, R anova(m0, m1) and AIC (stats 4.6.2) 6.66796668331; relative difference 7.99e-16 (allowed 1e-10)
      Source: R anova(m0, m1) and AIC (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/anova.lm.html
agrees  stackl

agrees  bioChemists (pscl) | compare_models (outcome="counts with many zeros") | statistic: simplydata 5.29698211687, R lrtest test of the two fits (stats 4.6.2) 5.29698211687; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lrtest test of the two fits (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/pscl/html/zeroinfl.html
agrees  bioChemists (pscl) | compare_models (outcome="counts with many zeros") | p-value: simplydata 0.0707579020861, R lrtest test of the two fits (stats 4.6.2) 0.0707579020861; relative difference 1.18e-15 (allowed 1e-06)
      Source: R lrtest test of the two fits (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/pscl/html/zeroinfl.html
Read "mammals.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  mammals (MASS) | compare_models (using= a list of models: AICc) | log_brain ~ log_body: AIC: simplydata 134.672927328, R AICcmodavg aictab (AICcmodavg 2.3.4) 134.672927328

## 11. Choosing columns with the lasso: `find_columns_that_predict`

The glmnet vignette's `QuickStartExample` and `BinomialExample`, against `cv.glmnet` after the same `set.seed`.

In [15]:
for name, outcome in (("QuickStartExample", "numbers"), ("BinomialExample", "yes or no")):
    data = sdata.table(cmp.data_file(name))
    columns = [c for c in quiet(data.get_column_names) if c != "y"]
    for test_method, seed in (("lasso", 1), ("elastic net", 1), ("lasso", 3)):
        if outcome == "yes or no" and test_method == "elastic net":
            continue
        t = Result(quiet(data.find_columns_that_predict, "y", using=columns, test_method=test_method, outcome=outcome, with_seed=seed), key="term")
        label = f'find_columns_that_predict (test_method="{test_method}", outcome="{outcome}", with_seed={seed})'
        kept = [c for c in columns if t[c, "effect"] != 0]
        ours = {"penalty chosen (lambda.1se)": t["penalty chosen (lambda.1se)", "penalty"], "columns kept (number)": len(kept)}
        ours.update({f"{c} effect": t[c, "effect"] for c in ["(intercept)"] + kept})
        cmp.check_all(f"{name} (glmnet)", label, ours)

Read "QuickStartExample.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  QuickStartExample (glmnet) | find_columns_that_predict (test_method="lasso", outcome="numbers", with_seed=1) | penalty chosen (lambda.1se): simplydata 0.159327079214, R glmnet cv.glmnet (same seed) (glmnet 5.2) 0.159327079214; relative difference 6.97e-16 (allowed 1e-12)
      Source: R glmnet cv.glmnet (same seed) (glmnet 5.2) (run reference) - https://search.r-project.org/CRAN/refmans/glmnet/html/cv.glmnet.html
agrees  QuickStartExample (glmnet) | find_columns_that_predict (test_method="lasso", outcome="numbers", with_seed=1) | columns kept (number): simplydata 8, R glmnet cv.glmnet (same seed) (glmnet 5.2) 8; relative difference 0.00e+00 (allowed 1e-12)
      Source: R glmnet cv.glmnet (same seed) (glmnet 5.2) (run reference) - https://search.r-project.org/CRAN/refmans/glmnet/html/cv.glmnet.html
agrees  QuickStartExample (glmnet) | find_columns_that_predict (test_method="lass

agrees  QuickStartExample (glmnet) | find_columns_that_predict (test_method="lasso", outcome="numbers", with_seed=3) | penalty chosen (lambda.1se): simplydata 0.145172900621, R glmnet cv.glmnet (same seed) (glmnet 5.2) 0.145172900621; relative difference 9.56e-16 (allowed 1e-12)
      Source: R glmnet cv.glmnet (same seed) (glmnet 5.2) (run reference) - https://search.r-project.org/CRAN/refmans/glmnet/html/cv.glmnet.html
agrees  QuickStartExample (glmnet) | find_columns_that_predict (test_method="lasso", outcome="numbers", with_seed=3) | columns kept (number): simplydata 8, R glmnet cv.glmnet (same seed) (glmnet 5.2) 8; relative difference 0.00e+00 (allowed 1e-12)
      Source: R glmnet cv.glmnet (same seed) (glmnet 5.2) (run reference) - https://search.r-project.org/CRAN/refmans/glmnet/html/cv.glmnet.html
agrees  QuickStartExample (glmnet) | find_columns_that_predict (test_method="lasso", outcome="numbers", with_seed=3) | (intercept) effect: simplydata 0.154739911019, R glmnet cv.glmn

agrees  BinomialExample (glmnet) | find_columns_that_predict (test_method="lasso", outcome="yes or no", with_seed=1) | penalty chosen (lambda.1se): simplydata 0.045060855538, R glmnet cv.glmnet (same seed) (glmnet 5.2) 0.045060855538; relative difference 2.00e-15 (allowed 1e-12)
      Source: R glmnet cv.glmnet (same seed) (glmnet 5.2) (run reference) - https://search.r-project.org/CRAN/refmans/glmnet/html/cv.glmnet.html
agrees  BinomialExample (glmnet) | find_columns_that_predict (test_method="lasso", outcome="yes or no", with_seed=1) | columns kept (number): simplydata 14, R glmnet cv.glmnet (same seed) (glmnet 5.2) 14; relative difference 0.00e+00 (allowed 1e-12)
      Source: R glmnet cv.glmnet (same seed) (glmnet 5.2) (run reference) - https://search.r-project.org/CRAN/refmans/glmnet/html/cv.glmnet.html
agrees  BinomialExample (glmnet) | find_columns_that_predict (test_method="lasso", outcome="yes or no", with_seed=1) | (intercept) effect: simplydata 0.20853753053, R glmnet cv.glm

## 12. The groups that pull a mixed model

`show_rows_that_pull_model(grouped_by="Subject")` gives back the subjects with a Cook's distance above 4 / 18, the
biggest first (each subject left out in turn and the model refitted). Compared with R `cooks.distance(influence(m,
groups = "Subject"))` (lme4) for the random-intercept model: the subjects found, in order, and their Cook's distances.

In [16]:
pulled = Result(quiet(sleep.show_rows_that_pull_model, "Reaction", using=["Days"], grouped_by="Subject"))
cmp.check_vector(D, "show_rows_that_pull_model (grouped_by=)", "Subject", pulled["Subject"])
cmp.check_vector(D, "show_rows_that_pull_model (grouped_by=)", "Cook's distance", pulled["Cook's distance"])

agrees  sleepstudy (lme4) | show_rows_that_pull_model (grouped_by=) | Subject: rows with the same value: simplydata 4, R lme4 influence (groups =) and cooks.distance (lme4 2.1.0) 4; difference 0.00e+00 (allowed 0e+00)
      Source: R lme4 influence (groups =) and cooks.distance (lme4 2.1.0) (run reference) - https://search.r-project.org/CRAN/refmans/lme4/html/influence.merMod.html
agrees  sleepstudy (lme4) | show_rows_that_pull_model (grouped_by=) | Cook's distance [1]: simplydata 0.467719327389, R lme4 influence (groups =) and cooks.distance (lme4 2.1.0) 0.467719327389; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lme4 influence (groups =) and cooks.distance (lme4 2.1.0) (run reference) - https://search.r-project.org/CRAN/refmans/lme4/html/influence.merMod.html
agrees  sleepstudy (lme4) | show_rows_that_pull_model (grouped_by=) | Cook's distance [2]: simplydata 0.339214602856, R lme4 influence (groups =) and cooks.distance (lme4 2.1.0) 0.339214602856; relative differen

In [17]:
cmp.finish()

26_models_mixed_and_more: 944 comparisons - 935 agrees, 8 explained difference – machine, 1 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,26_models_mixed_and_more,sleepstudy (lme4),"explain (grouped_by=, own_slope_for=""Days"")",(intercept) estimate,251.405105,251.410000,"Bates, Maechler, Bolker & Walker (2015), J. St...",-4.895152e-03,0.005000,absolute,4.895152e-03,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v067.i01 ; https:...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,26_models_mixed_and_more,sleepstudy (lme4),"explain (grouped_by=, own_slope_for=""Days"")",(intercept) standard error,6.824597,6.820000,"Bates, Maechler, Bolker & Walker (2015), J. St...",4.596695e-03,0.005000,absolute,4.596695e-03,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v067.i01 ; https:...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,26_models_mixed_and_more,sleepstudy (lme4),explain (grouped_by=),(intercept) estimate,251.405105,251.405105,R lme4 lmer (help page sleepstudy) (lme4 2.1.0),-2.842171e-14,0.000001,relative,1.130514e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lme4...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,26_models_mixed_and_more,sleepstudy (lme4),explain (grouped_by=),(intercept) standard error,9.746716,9.746716,R lme4 lmer (help page sleepstudy) (lme4 2.1.0),1.229167e-07,0.000001,relative,1.261109e-08,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lme4...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,26_models_mixed_and_more,sleepstudy (lme4),"explain (grouped_by=, p_value_by=""satterthwaite"")",(intercept) degrees of freedom,22.810199,22.810199,R lmerTest (Satterthwaite) (lmerTest 3.2.1),0.000000e+00,0.000001,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lmer...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,26_models_mixed_and_more,sleepstudy (lme4),show_rows_that_pull_model (grouped_by=),Subject: rows with the same value,4.000000,4.000000,R lme4 influence (groups =) and cooks.distance...,0.000000e+00,0.000000,absolute,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lme4...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
940,26_models_mixed_and_more,sleepstudy (lme4),show_rows_that_pull_model (grouped_by=),Cook's distance [1],0.467719,0.467719,R lme4 influence (groups =) and cooks.distance...,0.000000e+00,0.000001,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lme4...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
941,26_models_mixed_and_more,sleepstudy (lme4),show_rows_that_pull_model (grouped_by=),Cook's distance [2],0.339215,0.339215,R lme4 influence (groups =) and cooks.distance...,5.551115e-17,0.000001,relative,1.636461e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lme4...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
942,26_models_mixed_and_more,sleepstudy (lme4),show_rows_that_pull_model (grouped_by=),Cook's distance [3],0.290309,0.290309,R lme4 influence (groups =) and cooks.distance...,5.551115e-17,0.000001,relative,1.912140e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lme4...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
